In [0]:
from pyspark.sql import functions as F, Window

bronze = spark.table("workspace.bronze.empresas_raw")
print("linhas na bronze:", bronze.count())

In [0]:
# Visão geral da tabela bronze de empresas
print("linhas:", bronze.count(), "| colunas:", len(bronze.columns))
display(bronze)


In [0]:
#descartar colunas irrelevantes para a minha análise (guardei na camada bronze pq pode ser útil por algum motivo) Mas agora na camada silver não é necessário

COLUNAS_RUIDO = ["cor_marca", "codigo_legado", "flag_teste",
                 "obs_interna", "hash_antigo", "versao_layout"]
df = bronze.drop(*COLUNAS_RUIDO)
print(len(bronze.columns), "->", len(df.columns), "colunas")

In [0]:
# Limpeza básica de texto (em todas as colunas, menos metadados que começam com "_"):
#   1) TRIM: remove espaços no começo e no fim
#   2) Nulos disfarçados ("", "N/A", "N/D", "NULL", vistos no perfilamento) viram NULL de verdade

NULOS_DISFARCADOS = ["", "N/A", "N/D", "NULL"]

def limpa_texto(nome_coluna):
    t = F.trim(F.col(nome_coluna))
    return F.when(F.upper(t).isin(NULOS_DISFARCADOS), None).otherwise(t)

for c in df.columns:
    if not c.startswith("_"):
        df = df.withColumn(c, limpa_texto(c))

In [0]:
# Comparar o nulos por coluna depois do cód 4
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
           for c in df.columns if not c.startswith("_")]).show(vertical=True)

In [0]:
# Transformar os dados das colunas ticker, situacao e uf tudo em maiúsculas
df = df.withColumn("ticker", F.upper(F.col("ticker")))
df = df.withColumn("situacao", F.upper(F.col("situacao")))
df = df.withColumn("uf", F.upper(F.col("uf")))

#mostrar o resultado
display(df)




In [0]:
#Criar lista pas as colunas "setor", "subsetor", "cidade", "segmento_listagem", "nome_pregao") e padronizar os dados para que fiquem com a primeira letra de cada palavra em maiúscula

for c in ["setor", "subsetor", "cidade", "segmento_listagem", "nome_pregao"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

#mostrar o resultado
display(df)


In [0]:
#Tratar "razao_social". Não vou usar apenas initcap pois vai gerar problema com S.A final. Então vou usar initcap e depois substituir S.a por S.A

df = df.withColumn("razao_social", F.initcap(F.col("razao_social")))
df = df.withColumn("razao_social", F.regexp_replace(F.col("razao_social"), r"S\.a\.", "S.A."))

#mostrar o resultado
display(df)


In [0]:
# Validação: quantidade de valores distintos por coluna padronizada

for c in ["situacao", "setor", "subsetor", "segmento_listagem", "uf", "cidade"]:
    print(c, "->", df.select(c).distinct().count())

# Verificar se está tudo OK. Razões sociais que NÃO terminam em "S.A." (esperado: 0)
df.filter(~F.col("razao_social").endswith("S.A.")).count()

In [0]:
# TIPAGEM 1/4: id_empresa (string -> int)
# Por quê: nas cotações o id vem como '000123' e aqui como '123'. Como texto, o JOIN
# retorna 0 linhas. Com int os dois viram 123 e o JOIN funciona.
# try_cast devolve NULL em vez de derrubar a célula se algum id não for numérico.
df = df.withColumn("id_empresa", F.col("id_empresa").try_cast("int"))

# Validação
df.select("id_empresa").printSchema()                    
# esperado: integer
print(df.filter(F.col("id_empresa").isNull()).count())    
# esperado: 0 (nenhum id perdido na conversão)

In [0]:
# Tratar as duas colunas de datas
# Referência para a reconciliação: quantos "só ano" e quantos nulos existem ANTES da conversão
for c in ["data_fundacao", "data_listagem"]:
    so_ano = df.filter(F.col(c).rlike("^[0-9]{4}$")).count()
    nulos = df.filter(F.col(c).isNull()).count()
    print(c, "| só ano:", so_ano, "| nulos antes:", nulos)

In [0]:
# Datas chegam em 5 formatos. try_to_date devolve NULL se o formato não bater,
# e coalesce fica com o primeiro formato que funcionar.
# Defini a função parse_se  
def parse_data(col):
    return F.coalesce(
        F.expr(f"try_to_date({col}, 'dd/MM/yyyy')"),
        F.expr(f"try_to_date({col}, 'yyyy-MM-dd')"),
        F.expr(f"try_to_date({col}, 'dd-MM-yyyy')"),
        F.expr(f"try_to_date({col}, 'yyyyMMdd')"),
        F.expr(f"try_to_date({col}, 'dd.MM.yyyy')"),
    )